# Phase 03E: Teacher A ExpansionNet v2 Replacement Handoff

Notebook này hoàn tất việc thay **Teacher A CLIP ViT-B/32** bằng
**ExpansionNet v2** và phát hành đầu vào chính thức cho nhánh
**Teacher A → S1 MobileNetV3-Large bằng Sequence KD**.

## Mục tiêu

1. Xác minh toàn bộ chuỗi Phase 03B → 03C → 03D và checksum.
2. Kiểm định lại corpus 113.287 pseudo-caption mà không đọc ảnh.
3. Khóa quyết định thay CLIP bằng ExpansionNet v2 trên Validation.
4. Đóng gói Parquet/JSONL và hợp đồng Sequence KD tự chứa.
5. Cấp quyền cho Phase 04B replacement preflight.

## Kaggle inputs bắt buộc

1. Phase 2A Shared Student Pipeline.
2. Phase 03B ExpansionNet v2 Offline Preflight output.
3. Phase 03C ExpansionNet v2 Readiness output.
4. Phase 03D ExpansionNet v2 Full Pseudo-Caption output.

## Runtime

- Accelerator: **None/CPU**.
- Internet: **Off**.
- Chạy tuần tự Cell 1 → Cell 6.
- Không mở MS COCO images hoặc Held-out Test Parquet.

## Đầu ra dự kiến

`/kaggle/working//phase3_teacher_a_expansionnet_v2_handoff`


## Cell 1 — Khám phá input và khóa môi trường

**Công việc**

- tìm đúng một marker Phase 2A, 03B, 03C và 03D;
- xác nhận Teacher/Student, trạng thái và Test Set SEALED;
- tạo cây thư mục output.

**Đầu ra dự kiến:** `Status: PASSED`, tiến độ `1 / 6`.


In [1]:
import csv
import hashlib
import json
import os
import platform
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import pyarrow.parquet as pq


CELL_STARTED_AT = time.perf_counter()
INPUT_ROOT = Path(
    os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input")
)
OUTPUT_ROOT = Path(
    os.environ.get(
        "KLCN_OUTPUT_ROOT",
        "/kaggle/working//"
        "phase3_teacher_a_expansionnet_v2_handoff",
    )
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
RESULT_DIR = OUTPUT_ROOT / "results"
PSEUDOCAPTION_DIR = OUTPUT_ROOT / "pseudocaptions"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT,
    AUDIT_DIR,
    CONFIG_DIR,
    RESULT_DIR,
    PSEUDOCAPTION_DIR,
    MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def exactly_one(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, target)
    return target


PHASE2_READY_PATH = exactly_one(
    "PHASE2_SHARED_TRAINING_BUNDLE_READY.json"
)
PHASE3B_READY_PATH = exactly_one(
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_PREFLIGHT_READY.json"
)
PHASE3C_READY_PATH = exactly_one(
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_"
    "PSEUDOCAPTION_READINESS_READY.json"
)
PHASE3D_COMPLETE_PATH = exactly_one(
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_"
    "PSEUDOCAPTIONS_COMPLETE.json"
)

PHASE2_ROOT = PHASE2_READY_PATH.parent
PHASE3B_ROOT = PHASE3B_READY_PATH.parent
PHASE3C_ROOT = PHASE3C_READY_PATH.parent
PHASE3D_ROOT = PHASE3D_COMPLETE_PATH.parent

phase2_ready = read_json(PHASE2_READY_PATH)
phase3b_ready = read_json(PHASE3B_READY_PATH)
phase3c_ready = read_json(PHASE3C_READY_PATH)
phase3d_complete = read_json(PHASE3D_COMPLETE_PATH)

assert phase3b_ready["status"] == "ready"
assert phase3c_ready["status"] == "ready"
assert phase3d_complete["status"] == "complete"
assert phase3b_ready["teacher"] == "ExpansionNet v2"
assert phase3c_ready["teacher"] == "ExpansionNet v2"
assert phase3d_complete["teacher"] == "ExpansionNet v2"
assert phase3b_ready["matched_student"] == "S1 MobileNetV3-Large"
assert phase3c_ready["matched_student"] == "S1 MobileNetV3-Large"
assert phase3d_complete["matched_student"] == "S1 MobileNetV3-Large"
assert phase3d_complete["phase03e_handoff_authorized"] is True
assert phase3d_complete["sequence_kd_dataset_ready"] is True
for marker in [phase3b_ready, phase3c_ready, phase3d_complete]:
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False

cell01_audit = {
    "audit_version": "phase3e_expansionnet_cell01_v1",
    "created_at_utc": now_utc(),
    "status": "passed",
    "python": platform.python_version(),
    "phase2_marker": str(PHASE2_READY_PATH),
    "phase3b_marker": str(PHASE3B_READY_PATH),
    "phase3c_marker": str(PHASE3C_READY_PATH),
    "phase3d_marker": str(PHASE3D_COMPLETE_PATH),
    "teacher": "ExpansionNet v2",
    "matched_student": "S1 MobileNetV3-Large",
    "accelerator_required": False,
    "training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
}
write_json(AUDIT_DIR / "phase3e_cell01_input_discovery.json", cell01_audit)

print("=" * 72)
print("PHASE 03E EXPANSIONNET V2 CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher                   : A — ExpansionNet v2")
print("Matched Student           : S1 — MobileNetV3-Large")
print("Phase 2A marker           : FOUND")
print("Phase 03B / 03C / 03D     : FOUND / FOUND / FOUND")
print("Accelerator required      : False")
print("Training started          : False")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 6 code cells")
print("Next                      : verify upstream manifests and chain")


PHASE 03E EXPANSIONNET V2 CELL 1 STATUS
Status                    : PASSED
Teacher                   : A — ExpansionNet v2
Matched Student           : S1 — MobileNetV3-Large
Phase 2A marker           : FOUND
Phase 03B / 03C / 03D     : FOUND / FOUND / FOUND
Accelerator required      : False
Training started          : False
Test Set status           : SEALED
Progress                  : 1 / 6 code cells
Next                      : verify upstream manifests and chain


## Cell 2 — Xác minh manifest và chuỗi provenance

**Công việc**

- xác minh từng artifact trong manifest 03B, 03C và 03D;
- kiểm tra checkpoint SHA-256 xuyên suốt ba phase;
- kiểm tra hợp đồng compatibility và Sequence KD.

**Đầu ra dự kiến:** toàn bộ manifest `VERIFIED`, tiến độ `2 / 6`.


In [2]:
CELL_STARTED_AT = time.perf_counter()


def verify_manifest(marker, root, expected_rows):
    csv_ref = marker["manifest"]["csv"]
    sums_ref = marker["manifest"]["sha256sums"]
    csv_path = resolve_inside(root, csv_ref["relative_path"])
    sums_path = resolve_inside(root, sums_ref["relative_path"])
    assert sha256_file(csv_path) == csv_ref["sha256"]
    assert sha256_file(sums_path) == sums_ref["sha256"]
    with csv_path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == expected_rows
    for row in rows:
        artifact = resolve_inside(root, row["relative_path"])
        assert artifact.is_file(), artifact
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256_file(artifact) == row["sha256"]
    return rows


phase3b_rows = verify_manifest(phase3b_ready, PHASE3B_ROOT, 14)
phase3c_rows = verify_manifest(phase3c_ready, PHASE3C_ROOT, 11)
phase3d_rows = verify_manifest(phase3d_complete, PHASE3D_ROOT, 9)

checkpoint_sha = phase3b_ready["checkpoint_sha256"]
production_contract_ref = phase3c_ready["production_contract"]
production_contract_path = resolve_inside(
    PHASE3C_ROOT, production_contract_ref["relative_path"]
)
assert (
    sha256_file(production_contract_path)
    == production_contract_ref["sha256"]
)
production_contract = read_json(production_contract_path)
assert production_contract["teacher_checkpoint_sha256"] == checkpoint_sha
assert phase3d_complete["teacher_checkpoint_sha256"] == checkpoint_sha

for key in [
    "architecture_contract",
    "compatibility_contract",
    "sequence_kd_contract",
]:
    reference = phase3b_ready[key]
    artifact = resolve_inside(
        PHASE3B_ROOT, reference["relative_path"]
    )
    assert sha256_file(artifact) == reference["sha256"]

for key in [
    "production_contract",
    "token_mapping_contract",
    "shard_plan",
]:
    reference = phase3c_ready[key]
    artifact = resolve_inside(
        PHASE3C_ROOT, reference["relative_path"]
    )
    assert sha256_file(artifact) == reference["sha256"]

cell02_audit = {
    "audit_version": "phase3e_expansionnet_cell02_v1",
    "created_at_utc": now_utc(),
    "status": "passed",
    "phase3b_manifest_rows": len(phase3b_rows),
    "phase3c_manifest_rows": len(phase3c_rows),
    "phase3d_manifest_rows": len(phase3d_rows),
    "teacher_checkpoint_sha256": checkpoint_sha,
    "checkpoint_chain_exact": True,
    "architecture_contract_verified": True,
    "compatibility_contract_verified": True,
    "sequence_kd_contract_verified": True,
    "token_mapping_contract_verified": True,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
}
write_json(
    AUDIT_DIR / "phase3e_cell02_upstream_integrity.json",
    cell02_audit,
)

print("=" * 72)
print("PHASE 03E EXPANSIONNET V2 CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Phase 03B manifest        : 14 / 14 VERIFIED")
print("Phase 03C manifest        : 11 / 11 VERIFIED")
print("Phase 03D manifest        : 9 / 9 VERIFIED")
print("Checkpoint chain exact    : True")
print("KD contracts              : VERIFIED")
print("Token mapping contract    : VERIFIED")
print("Test Set status           : SEALED")
print("Progress                  : 2 / 6 code cells")
print("Next                      : semantic audit of pseudo-caption corpus")


PHASE 03E EXPANSIONNET V2 CELL 2 STATUS
Status                    : PASSED
Phase 03B manifest        : 14 / 14 VERIFIED
Phase 03C manifest        : 11 / 11 VERIFIED
Phase 03D manifest        : 9 / 9 VERIFIED
Checkpoint chain exact    : True
KD contracts              : VERIFIED
Token mapping contract    : VERIFIED
Test Set status           : SEALED
Progress                  : 2 / 6 code cells
Next                      : semantic audit of pseudo-caption corpus


## Cell 3 — Kiểm định corpus pseudo-caption

**Công việc**

- xác minh JSONL và Parquet theo marker 03D;
- kiểm tra 113.287 `global_index`, `image_id`, caption và token IDs;
- kiểm tra BOS/EOS, UNK, PAD, độ dài, checkpoint SHA và shard;
- không đọc ảnh hoặc caption Test.

**Đầu ra dự kiến:** corpus `PASSED`, tiến độ `3 / 6`.


In [3]:
CELL_STARTED_AT = time.perf_counter()

jsonl_ref = phase3d_complete["jsonl"]
parquet_ref = phase3d_complete["parquet"]
summary_ref = phase3d_complete["summary"]
SOURCE_JSONL = resolve_inside(
    PHASE3D_ROOT, jsonl_ref["relative_path"]
)
SOURCE_PARQUET = resolve_inside(
    PHASE3D_ROOT, parquet_ref["relative_path"]
)
SOURCE_SUMMARY = resolve_inside(
    PHASE3D_ROOT, summary_ref["relative_path"]
)
assert SOURCE_JSONL.stat().st_size == jsonl_ref["size_bytes"]
assert sha256_file(SOURCE_JSONL) == jsonl_ref["sha256"]
assert SOURCE_PARQUET.stat().st_size == parquet_ref["size_bytes"]
assert sha256_file(SOURCE_PARQUET) == parquet_ref["sha256"]
assert sha256_file(SOURCE_SUMMARY) == summary_ref["sha256"]

table = pq.read_table(SOURCE_PARQUET)
assert table.num_rows == 113287
required_columns = {
    "global_index",
    "image_id",
    "caption",
    "teacher_token_ids",
    "student_token_ids",
    "teacher_checkpoint_sha256",
    "shard_id",
}
assert required_columns.issubset(table.column_names)
data = table.select(sorted(required_columns)).to_pydict()

global_indices = data["global_index"]
image_ids = data["image_id"]
captions = data["caption"]
student_ids = data["student_token_ids"]
teacher_ids = data["teacher_token_ids"]
checkpoint_values = data["teacher_checkpoint_sha256"]
shard_ids = data["shard_id"]

assert global_indices == list(range(113287))
assert all(
    image_ids[index] < image_ids[index + 1]
    for index in range(len(image_ids) - 1)
)
assert len(set(image_ids)) == 113287
assert all(caption and caption.strip() for caption in captions)
assert all(ids[0] == 1 and ids[-1] == 2 for ids in student_ids)
assert all(0 not in ids and 3 not in ids for ids in student_ids)
assert all(ids[0] == 79 and ids[-1] == 77 for ids in teacher_ids)
assert max(len(ids) - 2 for ids in student_ids) <= 30
assert set(checkpoint_values) == {checkpoint_sha}
assert set(shard_ids) == {0}

with SOURCE_JSONL.open("r", encoding="utf-8") as handle:
    jsonl_count = sum(1 for line in handle if line.strip())
assert jsonl_count == 113287

summary = read_json(SOURCE_SUMMARY)
assert summary["records"] == 113287
assert summary["unique_image_ids"] == 113287
assert summary["unknown_tokens"] == 0
assert summary["student_bos_eos_valid"] is True
assert summary["validation_overlap"] == 0
assert summary["held_out_test_overlap"] == 0
assert summary["deterministic_spot_exact"] is True

cell03_audit = {
    "audit_version": "phase3e_expansionnet_cell03_v1",
    "created_at_utc": now_utc(),
    "status": "passed",
    "records": table.num_rows,
    "jsonl_records": jsonl_count,
    "unique_image_ids": len(set(image_ids)),
    "unique_captions": len(set(captions)),
    "image_order_exact": True,
    "student_bos_eos_valid": True,
    "unknown_tokens": 0,
    "maximum_lexical_tokens": max(
        len(ids) - 2 for ids in student_ids
    ),
    "validation_overlap": 0,
    "held_out_test_overlap": 0,
    "checkpoint_chain_exact": True,
    "shard_ids": [0],
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
}
write_json(
    AUDIT_DIR / "phase3e_cell03_pseudocaption_semantics.json",
    cell03_audit,
)

print("=" * 72)
print("PHASE 03E EXPANSIONNET V2 CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("JSONL / Parquet records   : 113287 / 113287")
print("Unique image IDs          : 113287")
print("Unique captions           :", len(set(captions)))
print("Image order exact         : True")
print("BOS/EOS valid             : True")
print("Unknown tokens            : 0")
print("Maximum lexical tokens    :", cell03_audit["maximum_lexical_tokens"])
print("Validation overlap        : 0")
print("Held-out Test overlap     : 0")
print("Test Set status           : SEALED")
print("Progress                  : 3 / 6 code cells")
print("Next                      : lock replacement decision")


PHASE 03E EXPANSIONNET V2 CELL 3 STATUS
Status                    : PASSED
JSONL / Parquet records   : 113287 / 113287
Unique image IDs          : 113287
Unique captions           : 60899
Image order exact         : True
BOS/EOS valid             : True
Unknown tokens            : 0
Maximum lexical tokens    : 17
Validation overlap        : 0
Held-out Test overlap     : 0
Test Set status           : SEALED
Progress                  : 3 / 6 code cells
Next                      : lock replacement decision


## Cell 4 — Khóa quyết định thay Teacher A

**Công việc**

- dùng cùng Karpathy Validation 5.000 ảnh;
- so sánh ExpansionNet v2 với Teacher A CLIP lịch sử;
- lưu bảng kết quả, chênh lệch tuyệt đối và tương đối;
- ghi rõ CE giữa hai Teacher không so sánh trực tiếp.

**Đầu ra dự kiến:** quyết định `LOCKED`, tiến độ `4 / 6`.


In [4]:
CELL_STARTED_AT = time.perf_counter()

new_metrics = phase3b_ready["validation_metrics"]
metrics_path = PHASE3B_ROOT / (
    "validation/expansionnet_v2_validation_metrics.json"
)
assert metrics_path.is_file()
metrics_payload = read_json(metrics_path)
assert metrics_payload["split"] == "Karpathy Validation"
assert metrics_payload["images"] == 5000
assert metrics_payload["references"] == 25010
old_clip_cider = float(
    metrics_payload["old_clip_teacher_a_cider_d_mean"]
)
new_cider = float(new_metrics["CIDEr-D"])
cider_delta = new_cider - old_clip_cider
cider_relative_gain = cider_delta / old_clip_cider
assert cider_delta > 0

comparison_rows = [
    {
        "teacher": "CLIP ViT-B/32 + shared decoder",
        "role": "historical Teacher A",
        "BLEU-1": "",
        "BLEU-4": "",
        "METEOR": "",
        "ROUGE-L": "",
        "CIDEr-D": f"{old_clip_cider:.12f}",
        "status": "historical_only",
    },
    {
        "teacher": "ExpansionNet v2",
        "role": "replacement Teacher A",
        "BLEU-1": f"{new_metrics['BLEU-1']:.12f}",
        "BLEU-4": f"{new_metrics['BLEU-4']:.12f}",
        "METEOR": f"{new_metrics['METEOR']:.12f}",
        "ROUGE-L": f"{new_metrics['ROUGE-L']:.12f}",
        "CIDEr-D": f"{new_cider:.12f}",
        "status": "selected",
    },
]
comparison_csv = RESULT_DIR / "teacher_a_replacement_comparison.csv"
with comparison_csv.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle, fieldnames=list(comparison_rows[0].keys())
    )
    writer.writeheader()
    writer.writerows(comparison_rows)

decision = {
    "contract_version": "teacher_a_expansionnet_replacement_v1",
    "created_at_utc": now_utc(),
    "status": "locked",
    "selection_split": "Karpathy Validation only",
    "validation_images": 5000,
    "validation_references": 25010,
    "previous_teacher": "CLIP ViT-B/32 + shared decoder",
    "replacement_teacher": "ExpansionNet v2",
    "matched_student": "S1 MobileNetV3-Large",
    "previous_teacher_cider_d_mean": old_clip_cider,
    "replacement_metrics": new_metrics,
    "cider_d_absolute_gain": cider_delta,
    "cider_d_relative_gain": cider_relative_gain,
    "replacement_reason": (
        "Higher CIDEr-D under the same locked Validation protocol"
    ),
    "previous_teacher_artifacts": "historical_only",
    "previous_logit_kd_results": "historical_only",
    "teacher_ce_comparison_allowed": False,
    "direct_logit_kd_allowed": False,
    "sequence_kd_required": True,
    "test_set_status": "sealed",
    "test_set_used": False,
}
DECISION_PATH = (
    CONFIG_DIR / "teacher_a_expansionnet_v2_replacement_decision.json"
)
write_json(DECISION_PATH, decision)

cell04_audit = {
    "audit_version": "phase3e_expansionnet_cell04_v1",
    "created_at_utc": now_utc(),
    "status": "passed",
    "old_clip_cider_d_mean": old_clip_cider,
    "expansionnet_v2_cider_d": new_cider,
    "absolute_gain": cider_delta,
    "relative_gain": cider_relative_gain,
    "decision": "ExpansionNet v2 replaces CLIP as Teacher A",
    "decision_locked": True,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
}
write_json(
    AUDIT_DIR / "phase3e_cell04_replacement_decision.json",
    cell04_audit,
)

print("=" * 72)
print("PHASE 03E EXPANSIONNET V2 CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Selection split           : Karpathy Validation")
print("Old CLIP CIDEr-D mean     :", round(old_clip_cider, 6))
print("ExpansionNet CIDEr-D      :", round(new_cider, 6))
print("Absolute gain             :", round(cider_delta, 6))
print("Relative gain             :", f"{100*cider_relative_gain:.2f}%")
print("Replacement decision      : LOCKED")
print("Old Logit KD              : HISTORICAL ONLY")
print("Direct Logit KD           : PROHIBITED")
print("Sequence KD               : REQUIRED")
print("Test Set status           : SEALED")
print("Progress                  : 4 / 6 code cells")
print("Next                      : publish self-contained Phase 4 handoff")


PHASE 03E EXPANSIONNET V2 CELL 4 STATUS
Status                    : PASSED
Selection split           : Karpathy Validation
Old CLIP CIDEr-D mean     : 1.160464
ExpansionNet CIDEr-D      : 1.397207
Absolute gain             : 0.236743
Relative gain             : 20.40%
Replacement decision      : LOCKED
Old Logit KD              : HISTORICAL ONLY
Direct Logit KD           : PROHIBITED
Sequence KD               : REQUIRED
Test Set status           : SEALED
Progress                  : 4 / 6 code cells
Next                      : publish self-contained Phase 4 handoff


## Cell 5 — Đóng gói handoff cho Phase 4

**Công việc**

- sao chép nguyên vẹn JSONL, Parquet và summary từ 03D;
- khóa loss `0,5 × CE_human + 0,5 × CE_sequence`;
- khóa image-balanced sampling và ba seed 42/123/2026;
- tạo registry và marker cấp quyền cho Phase 04B replacement preflight.

**Đầu ra dự kiến:** `Phase 04B AUTHORIZED`, tiến độ `5 / 6`.


In [5]:
CELL_STARTED_AT = time.perf_counter()

TARGET_JSONL = PSEUDOCAPTION_DIR / SOURCE_JSONL.name
TARGET_PARQUET = PSEUDOCAPTION_DIR / SOURCE_PARQUET.name
TARGET_SUMMARY = PSEUDOCAPTION_DIR / SOURCE_SUMMARY.name
for source, target in [
    (SOURCE_JSONL, TARGET_JSONL),
    (SOURCE_PARQUET, TARGET_PARQUET),
    (SOURCE_SUMMARY, TARGET_SUMMARY),
]:
    shutil.copy2(source, target)
    assert target.stat().st_size == source.stat().st_size
    assert sha256_file(target) == sha256_file(source)

compatibility_ref = phase3b_ready["compatibility_contract"]
sequence_ref = phase3b_ready["sequence_kd_contract"]
mapping_ref = phase3c_ready["token_mapping_contract"]
compatibility = read_json(
    resolve_inside(
        PHASE3B_ROOT, compatibility_ref["relative_path"]
    )
)
sequence_contract = read_json(
    resolve_inside(PHASE3B_ROOT, sequence_ref["relative_path"])
)
token_mapping = read_json(
    resolve_inside(PHASE3C_ROOT, mapping_ref["relative_path"])
)
assert compatibility["direct_logit_kd_allowed"] is False
assert compatibility["sequence_kd_required"] is True
assert token_mapping["validation_unknown_tokens"] == 0

handoff_contract = {
    "contract_version": "teacher_a_s1_sequence_kd_handoff_v1",
    "created_at_utc": now_utc(),
    "experiment_id": "P4A_EXPANSIONNET_S1_SEQKD",
    "teacher": "ExpansionNet v2",
    "teacher_mode": "frozen pretrained checkpoint",
    "student": "S1 MobileNetV3-Large",
    "student_initialization": "Phase 2A seed-specific initial checkpoint",
    "method": "sequence_kd",
    "seeds": [42, 123, 2026],
    "training_split": "Train+Restval only",
    "selection_split": "Karpathy Validation only",
    "held_out_test_policy": "sealed until final evaluation",
    "pseudo_caption_records": 113287,
    "pseudo_caption_parquet": {
        "relative_path": (
            "pseudocaptions/"
            "teacher_a_expansionnet_v2_train_restval.parquet"
        ),
        "size_bytes": TARGET_PARQUET.stat().st_size,
        "sha256": sha256_file(TARGET_PARQUET),
    },
    "pseudo_caption_jsonl": {
        "relative_path": (
            "pseudocaptions/"
            "teacher_a_expansionnet_v2_train_restval.jsonl"
        ),
        "size_bytes": TARGET_JSONL.stat().st_size,
        "sha256": sha256_file(TARGET_JSONL),
    },
    "target_conversion": (
        "teacher IDs -> words -> locked Phase 1 tokenizer -> "
        "student IDs with BOS/EOS"
    ),
    "loss": {
        "human_caption_ce_weight": 0.5,
        "teacher_sequence_ce_weight": 0.5,
        "formula": "L = 0.5 * L_human_CE + 0.5 * L_sequence_CE",
        "temperature": None,
    },
    "sampling": {
        "strategy": "image-balanced",
        "images_per_epoch": 113287,
        "human_captions_per_image_per_epoch": 1,
        "pseudo_captions_per_image": 1,
    },
    "direct_logit_kd_allowed": False,
    "feature_kd_in_phase4_allowed": False,
    "feature_kd_requires_new_phase5_projection": True,
    "checkpoint_selection_primary": "CIDEr-D",
    "tie_break_order": ["BLEU-4", "CE", "earlier epoch"],
    "test_set_status": "sealed",
}
HANDOFF_PATH = (
    CONFIG_DIR / "teacher_a_s1_sequence_kd_handoff_contract.json"
)
write_json(HANDOFF_PATH, handoff_contract)

registry = {
    "registry_version": "teacher_a_expansionnet_handoff_registry_v1",
    "created_at_utc": now_utc(),
    "teacher_checkpoint_sha256": checkpoint_sha,
    "source_phase3b_ready_sha256": sha256_file(PHASE3B_READY_PATH),
    "source_phase3c_ready_sha256": sha256_file(PHASE3C_READY_PATH),
    "source_phase3d_complete_sha256": sha256_file(
        PHASE3D_COMPLETE_PATH
    ),
    "replacement_decision": {
        "relative_path": str(DECISION_PATH.relative_to(OUTPUT_ROOT)),
        "sha256": sha256_file(DECISION_PATH),
    },
    "handoff_contract": {
        "relative_path": str(HANDOFF_PATH.relative_to(OUTPUT_ROOT)),
        "sha256": sha256_file(HANDOFF_PATH),
    },
    "pseudo_caption_parquet": handoff_contract[
        "pseudo_caption_parquet"
    ],
    "pseudo_caption_jsonl": handoff_contract[
        "pseudo_caption_jsonl"
    ],
    "test_set_status": "sealed",
}
REGISTRY_PATH = (
    CONFIG_DIR / "teacher_a_expansionnet_v2_artifact_registry.json"
)
write_json(REGISTRY_PATH, registry)

PHASE4_AUTH_PATH = (
    OUTPUT_ROOT
    / "PHASE4_TEACHER_A_S1_SEQUENCE_KD_PREFLIGHT_AUTHORIZED.json"
)
phase4_authorization = {
    "schema_version": "phase4_teacher_a_s1_seqkd_authorized_v1",
    "created_at_utc": now_utc(),
    "status": "authorized",
    "experiment_id": "P4A_EXPANSIONNET_S1_SEQKD",
    "teacher": "ExpansionNet v2",
    "student": "S1 MobileNetV3-Large",
    "selected_method": "sequence_kd",
    "seeds": [42, 123, 2026],
    "pseudo_caption_records": 113287,
    "handoff_contract": {
        "relative_path": str(HANDOFF_PATH.relative_to(OUTPUT_ROOT)),
        "sha256": sha256_file(HANDOFF_PATH),
    },
    "registry": {
        "relative_path": str(REGISTRY_PATH.relative_to(OUTPUT_ROOT)),
        "sha256": sha256_file(REGISTRY_PATH),
    },
    "next_phase": (
        "Phase 04B Teacher A to S1 Sequence KD replacement preflight"
    ),
    "training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(PHASE4_AUTH_PATH, phase4_authorization)

cell05_audit = {
    "audit_version": "phase3e_expansionnet_cell05_v1",
    "created_at_utc": now_utc(),
    "status": "passed",
    "copied_jsonl_exact": True,
    "copied_parquet_exact": True,
    "copied_summary_exact": True,
    "handoff_contract_created": True,
    "artifact_registry_created": True,
    "phase4_preflight_authorized": True,
    "selected_method": "sequence_kd",
    "human_ce_weight": 0.5,
    "teacher_sequence_ce_weight": 0.5,
    "seeds": [42, 123, 2026],
    "training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
}
write_json(
    AUDIT_DIR / "phase3e_cell05_phase4_handoff.json",
    cell05_audit,
)

print("=" * 72)
print("PHASE 03E EXPANSIONNET V2 CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Pseudo-caption JSONL      : COPIED EXACT")
print("Pseudo-caption Parquet    : COPIED EXACT")
print("Pseudo-caption records    : 113287")
print("Selected KD               : Sequence KD")
print("Human / Sequence CE       : 0.5 / 0.5")
print("Seeds                     : 42 / 123 / 2026")
print("Direct Logit KD           : PROHIBITED")
print("Phase 04B preflight       : AUTHORIZED")
print("Training started          : False")
print("Test Set status           : SEALED")
print("Progress                  : 5 / 6 code cells")
print("Next                      : manifest and final completion marker")


PHASE 03E EXPANSIONNET V2 CELL 5 STATUS
Status                    : PASSED
Pseudo-caption JSONL      : COPIED EXACT
Pseudo-caption Parquet    : COPIED EXACT
Pseudo-caption records    : 113287
Selected KD               : Sequence KD
Human / Sequence CE       : 0.5 / 0.5
Seeds                     : 42 / 123 / 2026
Direct Logit KD           : PROHIBITED
Phase 04B preflight       : AUTHORIZED
Training started          : False
Test Set status           : SEALED
Progress                  : 5 / 6 code cells
Next                      : manifest and final completion marker


## Cell 6 — Manifest, COMPLETE marker và tiến độ

**Công việc**

- lập manifest cho toàn bộ artifact handoff;
- xác minh lại từng checksum;
- phát hành COMPLETE marker;
- in tiến độ hiện tại và bước tiếp theo.

**Đầu ra dự kiến:** Phase 03E `PASSED`, tiến độ `6 / 6`.


In [6]:
CELL_STARTED_AT = time.perf_counter()

excluded_names = {
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_HANDOFF_COMPLETE.json",
}
manifest_rows = []
for artifact in sorted(OUTPUT_ROOT.rglob("*")):
    if not artifact.is_file():
        continue
    if artifact.name in excluded_names:
        continue
    if artifact.parent == MANIFEST_DIR:
        continue
    manifest_rows.append(
        {
            "relative_path": artifact.relative_to(OUTPUT_ROOT).as_posix(),
            "size_bytes": artifact.stat().st_size,
            "sha256": sha256_file(artifact),
        }
    )

MANIFEST_CSV = (
    MANIFEST_DIR / "phase3e_expansionnet_v2_handoff_manifest.csv"
)
with MANIFEST_CSV.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)

SHA256SUMS = MANIFEST_DIR / "SHA256SUMS"
SHA256SUMS.write_text(
    "".join(
        f"{row['sha256']}  {row['relative_path']}\n"
        for row in manifest_rows
    ),
    encoding="utf-8",
)

with MANIFEST_CSV.open(
    "r", encoding="utf-8", newline=""
) as handle:
    verification_rows = list(csv.DictReader(handle))
assert len(verification_rows) == len(manifest_rows)
for row in verification_rows:
    artifact = resolve_inside(OUTPUT_ROOT, row["relative_path"])
    assert artifact.is_file()
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

complete_marker = {
    "schema_version": "phase3e_expansionnet_handoff_complete_v1",
    "created_at_utc": now_utc(),
    "status": "complete",
    "phase": "03E",
    "teacher": "ExpansionNet v2",
    "matched_student": "S1 MobileNetV3-Large",
    "replacement_decision": "locked",
    "old_teacher": "CLIP ViT-B/32 + shared decoder",
    "old_teacher_status": "historical_only",
    "selected_method": "sequence_kd",
    "pseudo_caption_records": 113287,
    "pseudo_caption_parquet_sha256": sha256_file(TARGET_PARQUET),
    "direct_logit_kd_allowed": False,
    "phase4_preflight_authorized": True,
    "phase4_authorization": {
        "relative_path": PHASE4_AUTH_PATH.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "sha256": sha256_file(PHASE4_AUTH_PATH),
    },
    "manifest": {
        "csv": {
            "relative_path": MANIFEST_CSV.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(MANIFEST_CSV),
            "rows": len(manifest_rows),
        },
        "sha256sums": {
            "relative_path": SHA256SUMS.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(SHA256SUMS),
        },
    },
    "training_started": False,
    "test_parquet_opened": False,
    "test_caption_read": False,
    "test_images_read": False,
    "test_metrics_computed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
COMPLETE_PATH = (
    OUTPUT_ROOT
    / "PHASE3_TEACHER_A_EXPANSIONNET_V2_HANDOFF_COMPLETE.json"
)
write_json(COMPLETE_PATH, complete_marker)

print("=" * 72)
print("PHASE 03E EXPANSIONNET V2 FINAL STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher A                 : ExpansionNet v2")
print("Replaces                  : CLIP ViT-B/32 + shared decoder")
print("Matched Student           : S1 — MobileNetV3-Large")
print("Replacement decision      : LOCKED")
print("Pseudo-caption records    : 113287")
print("JSONL / Parquet           : READY / READY")
print("Selected method           : Sequence KD")
print("Direct Logit KD           : PROHIBITED")
print("Manifest rows             :", len(manifest_rows), "VERIFIED")
print("Phase 04B preflight       : AUTHORIZED")
print("Training started          : False")
print("Test Set status           : SEALED")
print("COMPLETE marker           :", COMPLETE_PATH)
print("COMPLETE SHA-256          :", sha256_file(COMPLETE_PATH))
print("Progress                  : 6 / 6 code cells")
print(
    "Next                      : Phase 04B Teacher A → S1 "
    "Sequence KD replacement preflight"
)


PHASE 03E EXPANSIONNET V2 FINAL STATUS
Status                    : PASSED
Teacher A                 : ExpansionNet v2
Replaces                  : CLIP ViT-B/32 + shared decoder
Matched Student           : S1 — MobileNetV3-Large
Replacement decision      : LOCKED
Pseudo-caption records    : 113287
JSONL / Parquet           : READY / READY
Selected method           : Sequence KD
Direct Logit KD           : PROHIBITED
Manifest rows             : 13 VERIFIED
Phase 04B preflight       : AUTHORIZED
Training started          : False
Test Set status           : SEALED
COMPLETE marker           : /kaggle/working//phase3_teacher_a_expansionnet_v2_handoff/PHASE3_TEACHER_A_EXPANSIONNET_V2_HANDOFF_COMPLETE.json
COMPLETE SHA-256          : 4fbbbb575c15985fe7bc487fd10a295398768b9bead4754c4a102aa8a8d71768
Progress                  : 6 / 6 code cells
Next                      : Phase 04B Teacher A → S1 Sequence KD replacement preflight
